# ISOLATION FOREST - POLLUTION ANOMALY DETECTION




In [1]:
import pandas as pd
import numpy as np
import pickle
import os

from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler

## Configuration

In [2]:
DATA_PATH = "../data/city_hour.csv"
MODEL_SAVE_PATH = "../models/isolation_forest.pkl"
SCALER_SAVE_PATH = "../models/iso_scaler.pkl"

feature_cols = [
    'PM2.5',
    'NO2',
    'CO',
    'SO2',
    'O3'
]

## Load Data

In [3]:
print("Loading hourly dataset...")
df = pd.read_csv(DATA_PATH)

df['Datetime'] = pd.to_datetime(df['Datetime'])
df = df.sort_values(['City', 'Datetime']).reset_index(drop=True)

print("Initial shape:", df.shape)

Loading hourly dataset...
Initial shape: (707875, 16)


## Handle Missing Values

In [4]:
print("Handling missing values...")

df = df.groupby('City', group_keys=False).apply(lambda x: x.ffill().bfill())

df = df.dropna(subset=feature_cols)

print("Shape after cleaning:", df.shape)

Handling missing values...
Shape after cleaning: (704023, 15)


## Feature Selection and Scaling

In [5]:
X = df[feature_cols]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Features scaled.")

Features scaled.


## Train Isolation Forest

In [6]:
print("Training Isolation Forest...")

iso_model = IsolationForest(
    n_estimators=200,
    contamination=0.02,
    random_state=42,
    n_jobs=-1
)

iso_model.fit(X_scaled)

print("Training complete.")

Training Isolation Forest...
Training complete.


## Save Model

In [7]:
os.makedirs("../models", exist_ok=True)

with open(MODEL_SAVE_PATH, "wb") as f:
    pickle.dump(iso_model, f)

with open(SCALER_SAVE_PATH, "wb") as f:
    pickle.dump(scaler, f)

print("Isolation Forest model saved successfully.")

Isolation Forest model saved successfully.


## Evaluation

In [8]:
preds = iso_model.predict(X_scaled)

anomaly_count = np.sum(preds == -1)
total_count = len(preds)

print("Total samples:", total_count)
print("Anomalies detected:", anomaly_count)
print("Anomaly %:", anomaly_count / total_count)

scores = iso_model.decision_function(X_scaled)

print("Score min:", scores.min())
print("Score max:", scores.max())

Total samples: 704023
Anomalies detected: 14081
Anomaly %: 0.02000076702039564
Score min: -0.16390985764732335
Score max: 0.28078941863920154
